#### Authenticate, clone the repo and setup the environment

In [ ]:
from pathlib import Path

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL: str = f"https://{token}@github.com/Krisenberg/impact.git"
BRANCH: str = "main"
REPO_DIR: str = "impact"

%cd /kaggle/working

if Path(REPO_DIR).is_dir():
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
%pip install uv -q

# '--extra cu128' will install CUDA 12.8 accelerator for PyTorch
!uv sync --no-dev --extra cu128

#### Verify that GPU is available

In [ ]:
import torch

device_name = torch.cuda.get_device_name(0)
device_memory = torch.cuda.get_device_properties(0).total_memory // 1e9
print(f"{device_name} | VRAM: {device_memory} GB")

#### Set the environment variable(s)

In [ ]:
import os
from typing import Any

env_vars: dict[str, Any] = {
    "HF_TOKEN": UserSecretsClient().get_secret("HF_TOKEN"),
}

os.environ.update(env_vars)
print("Environment variables successfully updated")

#### Train SASRecNBR (Yambda-50M dataset)

In [ ]:
!uv run cli train-sasrec \
    --dataset-size 50m \
    --sequence-step 5 \
    --sasrec-embedding-dim 128 \
    --sasrec-num-heads 2 \
    --sasrec-num-layers 2 \
    --sasrec-dropout-rate 0.2 \
    --sasrec-learning-rate 0.001

#### Train SASRecNBR (Yambda-5B dataset)

**Note:** The raw Yambda-5B dataset exceeds Kaggle's notebook storage limit (19.5 GB, where `multi_event.parquet` with raw data is 38.2 GB), so the data pipeline cannot be run directly in this environment. Instead, the dataset was pre-processed locally and uploaded as a Kaggle Dataset, mounted in the notebook at `/kaggle/input/datasets/krzysztofglowacz/yambda-5b-processed`. The `--processed-data-dir` flag points the pipeline to the processed files, bypassing raw data fetching and processing entirely.

In [ ]:
!uv run cli train-sasrec \
    --dataset-size 5b \
    --processed-data-dir /kaggle/input/datasets/krzysztofglowacz/yambda-5b-processed \
    --sequence-step 20 \
    --sasrec-train-batch-size 2048 \
    --sasrec-eval-batch-size 256 \
    --sasrec-embedding-dim 128 \
    --sasrec-num-heads 2 \
    --sasrec-num-layers 2 \
    --sasrec-dropout-rate 0.2 \
    --sasrec-learning-rate 0.001